In [2]:
from langgraph.graph import StateGraph , START ,END
from typing import TypedDict ,Annotated
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

In [3]:
load_dotenv()
llm = ChatGroq(model="llama-3.1-8b-instant" , temperature= 0.8)

In [4]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation : str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpoint = InMemorySaver()
workflow = graph.compile(checkpointer= checkpoint)

In [7]:
config1 = {"configurable":{"thread_id":1}}
workflow.invoke({"topic":"moon"} , config=config1)

{'topic': 'moon',
 'joke': 'Why did the astronaut break up with his girlfriend before going to the moon? \n\nBecause he needed space.',
 'explanation': 'The joke is a play on words. "Space" has a double meaning here: \n\n1. The astronaut\'s profession involves traveling to outer space, literally the vacuum of space outside the Earth\'s atmosphere.\n2. The phrase "needing space" is a common idiomatic expression that means requiring some time or distance from someone to think, reflect, or cope with emotions. It\'s often used in the context of relationships, implying that someone needs to be alone or separated from their partner.\n\nThe punchline of the joke is funny because it takes the literal meaning of "space" (the astronaut\'s work) and connects it to the idiomatic expression "needing space" (in a relationship). This unexpected twist creates a clever and humorous connection between the setup and the punchline.'}

In [8]:
workflow.get_state(config=config1) #final state value

StateSnapshot(values={'topic': 'moon', 'joke': 'Why did the astronaut break up with his girlfriend before going to the moon? \n\nBecause he needed space.', 'explanation': 'The joke is a play on words. "Space" has a double meaning here: \n\n1. The astronaut\'s profession involves traveling to outer space, literally the vacuum of space outside the Earth\'s atmosphere.\n2. The phrase "needing space" is a common idiomatic expression that means requiring some time or distance from someone to think, reflect, or cope with emotions. It\'s often used in the context of relationships, implying that someone needs to be alone or separated from their partner.\n\nThe punchline of the joke is funny because it takes the literal meaning of "space" (the astronaut\'s work) and connects it to the idiomatic expression "needing space" (in a relationship). This unexpected twist creates a clever and humorous connection between the setup and the punchline.'}, next=(), config={'configurable': {'thread_id': '1', 

In [9]:
list(workflow.get_state_history(config=config1)) #value of each state

[StateSnapshot(values={'topic': 'moon', 'joke': 'Why did the astronaut break up with his girlfriend before going to the moon? \n\nBecause he needed space.', 'explanation': 'The joke is a play on words. "Space" has a double meaning here: \n\n1. The astronaut\'s profession involves traveling to outer space, literally the vacuum of space outside the Earth\'s atmosphere.\n2. The phrase "needing space" is a common idiomatic expression that means requiring some time or distance from someone to think, reflect, or cope with emotions. It\'s often used in the context of relationships, implying that someone needs to be alone or separated from their partner.\n\nThe punchline of the joke is funny because it takes the literal meaning of "space" (the astronaut\'s work) and connects it to the idiomatic expression "needing space" (in a relationship). This unexpected twist creates a clever and humorous connection between the setup and the punchline.'}, next=(), config={'configurable': {'thread_id': '1',

GOING TO PREVIOUS STATE

In [10]:
workflow.get_state({"configurable":{"thread_id":1 , "checkpoint_id":"1f19232b-b19b-6d5e-8000-8c24d5dc45ff"}})

StateSnapshot(values={'topic': 'moon'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f19232b-b19b-6d5e-8000-8c24d5dc45ff'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-07T07:36:41.024221+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19232b-b198-636c-bfff-6bdd9102b6ed'}}, tasks=(PregelTask(id='0f6231b9-eb8d-8f16-094a-a56652795ff1', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the astronaut break up with his girlfriend before going to the moon? \n\nBecause he needed space.'}),), interrupts=())

In [11]:
workflow.invoke(None ,{"configurable":{"thread_id":1 , "checkpoint_id":"1f19232b-b19b-6d5e-8000-8c24d5dc45ff"}})

#none => no new initial_state(resume)

{'topic': 'moon',
 'joke': 'Why did the astronaut break up with his girlfriend before going to the moon?\n\nBecause he needed space.',
 'explanation': 'This joke is a play on words. "Needed space" has a double meaning here. \n\n1. The literal meaning: The astronaut is going to the moon, a celestial body in outer space. He is physically moving into a space that is far away from his girlfriend\'s gravitational pull, leaving her behind.\n\n2. The idiomatic meaning: "Needed space" is also a common idiomatic expression that means someone needs time and distance from a relationship to feel liberated or free. In this context, the astronaut is breaking up with his girlfriend because he feels he needs time and emotional space to focus on his own journey and mission to the moon.\n\nThe joke relies on this wordplay to create a clever and funny connection between the literal and idiomatic meanings of "needed space".'}

In [12]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'moon', 'joke': 'Why did the astronaut break up with his girlfriend before going to the moon?\n\nBecause he needed space.', 'explanation': 'This joke is a play on words. "Needed space" has a double meaning here. \n\n1. The literal meaning: The astronaut is going to the moon, a celestial body in outer space. He is physically moving into a space that is far away from his girlfriend\'s gravitational pull, leaving her behind.\n\n2. The idiomatic meaning: "Needed space" is also a common idiomatic expression that means someone needs time and distance from a relationship to feel liberated or free. In this context, the astronaut is breaking up with his girlfriend because he feels he needs time and emotional space to focus on his own journey and mission to the moon.\n\nThe joke relies on this wordplay to create a clever and funny connection between the literal and idiomatic meanings of "needed space".'}, next=(), config={'configurable': {'thread_id': '1', 'checkp

In [15]:
#4 states => from prev(complete flow)
#next 3 states =>from going to prev. state(fork)
#therefore total 7 states

UPDATING STATE

In [13]:
workflow.update_state({"configurable":{"thread_id":1 , "checkpoint_id":"1f19232b-b19b-6d5e-8000-8c24d5dc45ff","checkpoint_ns": ""}} 
                      , {'topic':'money'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f19232f-d569-644e-8001-15acd9ca5fd4'}}

In [14]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'money'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19232f-d569-644e-8001-15acd9ca5fd4'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-08-07T07:38:32.152562+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19232b-b19b-6d5e-8000-8c24d5dc45ff'}}, tasks=(PregelTask(id='cd485b28-9d25-542e-c238-7185b3b8cc4f', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'moon', 'joke': 'Why did the astronaut break up with his girlfriend before going to the moon?\n\nBecause he needed space.', 'explanation': 'This joke is a play on words. "Needed space" has a double meaning here. \n\n1. The literal meaning: The astronaut is going to the moon, a celestial body in outer space. He is physically moving into a space tha

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f19232f-d569-644e-8001-15acd9ca5fd4"}})
#checkpoint_id=> of  current updated state

{'topic': 'money',
 'joke': 'Why did the dollar bill go to therapy? \n\nBecause it was feeling a little flat and struggling to make change in its life.',
 'explanation': 'The joke relies on a play on words, using puns to create a humorous connection between the situation and the punchline.\n\nHere\'s a breakdown of the explanation:\n\n1. **Initial setup**: The joke starts with the premise that a dollar bill went to therapy. This is an unexpected twist, as dollar bills are inanimate objects and can\'t undergo therapy. The unexpected twist piques the listener\'s interest and creates a sense of curiosity.\n2. **Common issues**: The joke then states that the dollar bill was feeling "a little flat." This phrase has a double meaning:\n\t* A dollar bill can be physically flat, which is a literal description of its paper form.\n\t* The phrase "feeling flat" is also an idiomatic expression meaning feeling unenthusiastic or depressed.\n3. **Therapy context**: In a therapy setting, patients typic

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'money', 'joke': 'Why did the dollar bill go to therapy? \n\nBecause it was feeling a little flat and struggling to make change in its life.', 'explanation': 'The joke relies on a play on words, using puns to create a humorous connection between the situation and the punchline.\n\nHere\'s a breakdown of the explanation:\n\n1. **Initial setup**: The joke starts with the premise that a dollar bill went to therapy. This is an unexpected twist, as dollar bills are inanimate objects and can\'t undergo therapy. The unexpected twist piques the listener\'s interest and creates a sense of curiosity.\n2. **Common issues**: The joke then states that the dollar bill was feeling "a little flat." This phrase has a double meaning:\n\t* A dollar bill can be physically flat, which is a literal description of its paper form.\n\t* The phrase "feeling flat" is also an idiomatic expression meaning feeling unenthusiastic or depressed.\n3. **Therapy context**: In a therapy set

FAULT TORELANCE

In [3]:
import time

In [4]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(10)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}  

In [5]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [7]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [8]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [9]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19234d-15df-6576-8008-ed22e8493d52'}}, metadata={'source': 'loop', 'step': 8, 'parents': {}}, created_at='2026-08-07T07:51:37.374641+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19234d-15dc-6be1-8007-5b59c88a67cd'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19234d-15dc-6be1-8007-5b59c88a67cd'}}, metadata={'source': 'loop', 'step': 7, 'parents': {}}, created_at='2026-08-07T07:51:37.373565+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19234b-b995-6946-8006-f9e3bc0529e7'}}, tasks=(PregelTask(id='1b7ed37c-6843-e835-80af-d28039556a7